In [ ]:
# 1
Por que não ValueError? ValueError é genérico e já é tratado pelo FastAPI como 422. ContatoNaoEncontradoError é uma condição de negócio específica e merece seu próprio tipo para que o handler possa retornar 404 com mensagem clara.

Por que não ParseError? ParseError semânticamente indica falha de estrutura HTML/JSON. "Contato não encontrado" é uma resposta válida e esperada do LegalOne — não é um erro de parsing.

# 2
Mapeamento de nomes de contexto para IDs internos da API do LegalOne.
Detalhe de implementação da API — não exposto fora deste módulo.
_CONTEXT_IDS: dict[str, str] = {
    "Processos": "1",
    "Contatos":  "3",
}

# 3 
No router contacts:
- Tem Conversor schema → domain types 
- Tem definicao de funcao Dependency
- revisar arquivo dependencies.py em /app

# 4
testes de contacts e search

# 5
no service contact:
- Mover para infrastructure.lookup.select_mapper os atributos
_FN_TAG        = "Tag_PessoaFisicaEntitySchema_p3699_o"
_FN_LINK_DRIVE = "LinkDaPasta_PessoaFisicaEntitySchema_p3700_o"
_FN_DT_KIT     = "DataVencimentoKit_PessoaFisicaEntitySchema_p3701_o"
_FN_DT_COMP    = "DataVencimentoComprovante_PessoaFisicaEntitySchema_p3702_o"
_FN_CID        = "CID_PessoaFisicaEntitySchema_p3706_o"
_FN_REFERENCIA = "Referencia_PessoaFisicaEntitySchema_p3716_o"

# 6
Incluir no body de response de cadastros o id gerado pelo LegalOne

# 7
ajuste a rota de consulta de processos por cpf. quero que seja um GET com parametro um termo que pode ser o nome ou CPF do contato. ajuste o nome semantico da rota também que melhor reflita o que ela faz. 

# 8
implementar interface no metodo lookup_by_cpf do crawler contacts

# 9 
revisar onde deve ficar a funcao parse_list_of_lawsuits atualmente em services/lawsuit/lawsuit_service.py. Talvez deva ser movida para infrastructure.crawler.lawsuits ou para um módulo de parsing específico.

# 10
revisar localizacao dos objetos de négicio e DTOs e onde devem ficar.

# 11
- renomear service e crawler de pesquisa global para GlobalSearchService e GlobalSearchCrawler

# 12
revisar estrutura das rotas de listagem de modo que permita novos termos filtro 

# 13
tem objetos dtos em service contacts que sao na verdade dtos dos crawlers
lookup_grid_contact em contactsService está fazendo parsing desses objetos (verificar se é mesmo objeto de crawler ou service)

# 14
CreateTaskResponse deve ter consistencia com CreateContactResponse

# 15
Nos schemas, se quiser aceitar None convertendo para "":
from pydantic import field_validator
from typing import Any

class Exemplo(BaseModel):
    campo: str = ""

    @field_validator("campo", mode="before")
    @classmethod
    def none_para_vazio(cls, v: Any) -> str:
        return "" if v is None else v

# 16
Inconsistencia atributos errors e warnings em CreateContactResult (service) e schema CreateContactResponse. No service, errors e warnings são listas de strings. No schema, são listas de objetos com campo message. Padronizar ambos para lista de strings para simplificar.

# 17
NÃO cadastrado Errors: {'detail': [{'type': 'value_error', 'loc': ['body', 'endereco', 'uf'], 'msg': "Value error, UF deve conter exatamente 2 letras (ex.: 'SP').", 'input': 'PARQUE BRISTOL', 'ctx': {'error': {}}}]}


In [ ]:
# nomes tarefas automacao emails INSS:

1. tarefa de concessao
- nome: ANÁLISE DE CONCESSÃO ADMINISTRATIVA - BACKOFFICE
- responsável: camila vieira

2. avisar pericia
- nome: AVISAR CLIENTE PERÍCIA ADMINISTRATIVA
- resp: luana

3. conversao
- nome: CONVERSÃO REQUERIMENTO ADMINISTRATIVO EM JUDICIAL
- resp: gabriella

4. cumprimento de exigência
- nome: CUMPRIMENTO DE EXIGÊNCIA ADMINISTRATIVA
- resp: vitória

5. análise de caso (indeferimento por razoes variadas ou email indefinido)
- nome: ANÁLISE DE CASO - BACKOFFICE
- resp: gabriella

6. emails de mera ciencia
- nome: ANÁLISE DE EMAIL DO INSS SEM PRAZO
- resp: gabriella


# Kanban
- quadros:
    - BACKOFFICE
        - colunas:
            - A DESIGNAR
            - CONCESSÃO ADMINISTRATIVA # para concessao
            - INDEFERIMENTO ADM. # conversao e analise de caso
            - CUMPRIMENTO DE EXIGÊNGIA # demais cumprimentos de exigencia
            - 

    - RELACIONAMENTO | ACIDENTARIO
        - colunas:
            - A DESIGNAR
            - PERICIA ADMINISTRATIVA # para avisar pericia


# posicao
- Final da coluna # padrao

# status
- Pendente # padrao

# prioridade
- Normal # padrao

# tipo
- Diversos # padrao

# data previsto
- a data e hora de criação da tarefa

# data conclusao
- se for exigência, a data e hora prevista é 25 dias da data de criação da tarefa, incluindo o dia do começo
- se for perícia, a data e hora da perícia
- dia e hora seguinte da data de criação da tarefa # padrao

# prazo fatal
- se for perícia, é a data da perícia.
- se for outras exigências: 25 dias da data de criacao da tarefa, incluindo o dia do começo
- nos demais casos, sem prazo fatal.

# envolvimentos
- o responsável (Responsável e Executante)
    - campos:
        - Solicitante False
        - Responsável True
        - Executante True

; # lembrete
; - Notificar: Com antecedência da conclusão
; - 1 dia(s)
; - Notificação

# tarefas pendentes ADVBOX
- data e hora de inicio e conclusao no legalone, vai ser data e hora de inicio no advbox.
- prazo fatal no legalone vai ser igual ao prazo fatal no advbox, se tiver, se nao, fica sem prazo fatal.


In [ ]:
na etapa 4, quero usar SSM Parameter Store.
em relacao à camada de dependencia, quero usar docker para empacotar a aplicação.
o runtime pode ser de 60s e MemorySize: 512 MB.



In [ ]:
cd /Users/thomas/Documents/projetos/legal-one-client && sam build --no-cached 2>&1 | tail -8

# Smoke Tests
```sh
API_URL="https://vtyx2rgl2d.execute-api.us-east-1.amazonaws.com/prod"

API_KEY=$(aws ssm get-parameter --name /legal-one/api-key --with-decryption --query Parameter.Value --output text) && \

TOKEN=$(curl -s -X POST "$API_URL/auth/token" -H "Content-Type: application/json" -d "{\"api_key\":\"$API_KEY\"}" | python3 -c "import sys,json; d=json.load(sys.stdin); print(d.get('access_token','ERRO: '+str(d)))") && \

echo "" && echo "Token: ${TOKEN:0:60}..." && echo "" && \
curl -s -w "\nHTTP %{http_code}\n" "$API_URL/contacts/12345678900" -H "Authorization: Bearer $TOKEN" 2>&1

curl -s -w "\nHTTP %{http_code}\n" "$API_URL/docs" -H "Authorization: Bearer $TOKEN" 2>&1

echo "" && echo "=== 5. GET /contacts/ (com token — espera 404 ou dados) ===" && echo ""
curl -s "$API_URL/contacts/654.873.627-34" \ 
    -H "Authorization: Bearer $TOKEN" -w "\nHTTP %{http_code}\n"
```

# Comando para verifica como as classes foram declaradas no novo arquivo
```sh
cd /Users/thomas/Documents/projetos/legal-one-client && grep -n "^class " tests/contacts/test_lookup_by_cpf.py
```

# Abrir Docker via Terminal
```sh
open -a Docker && sleep 8 && docker info > /dev/null 2>&1 && echo "Docker OK"
```
- O comando `open -a Docker` é usado para abrir o aplicativo Docker no macOS. Ele inicia o Docker Desktop, que é a interface gráfica para gerenciar contêineres Docker.
- O comando `sleep 8` é usado para pausar a execução do script por 8 segundos, dando tempo para o Docker Desktop iniciar completamente antes de tentar verificar seu status.
- O comando `docker info > /dev/null 2>&1` é usado para verificar se o Docker está funcionando corretamente. Ele executa o comando `docker info`, que exibe informações sobre a instalação do Docker. A parte `> /dev/null 2>&1` redireciona a saída padrão (stdout) e a saída de erro (stderr) para o dispositivo nulo, o que significa que qualquer saída do comando será descartada. Se o comando for bem-sucedido, ele não produzirá nenhuma saída visível, mas se houver um problema com o Docker, ele pode falhar silenciosamente. O comando `&& echo "Docker OK"` é usado para imprimir "Docker OK" se o comando `docker info` for bem-sucedido, indicando que o Docker está funcionando corretamente.

# Comandos Logs AWS
```sh
aws logs tail /aws/lambda/legalone-client --since 5m --format short 2>&1 | tail -20
aws logs filter-log-events --log-group-name /aws/lambda/legalone-client --filter-pattern "_retry_transient" 2>/dev/null | tail -20
```
- O comando `aws logs tail` é usado para seguir os logs de uma função Lambda específica, neste caso, a função `legalone-client`. A opção `--since 5m` indica que queremos ver os logs dos últimos 5 minutos, e `--format short` formata a saída de forma mais concisa. O `2>&1` redireciona a saída de erro padrão (stderr) para a saída padrão (stdout), garantindo que todas as mensagens sejam capturadas. O comando `| tail -20` é usado para mostrar apenas as últimas 20 linhas da saída, o que pode ser útil para focar nas mensagens mais recentes.
- tail do comando aws é difente do tail do final do comando de linha, pois o tail do aws logs é uma funcionalidade específica do AWS CLI que permite seguir os logs em tempo real, enquanto o tail no final do comando é um utilitário de linha de comando tradicional que exibe as últimas linhas de uma saída. O `aws logs tail` é projetado para funcionar com os serviços de log da AWS, enquanto o `tail` é uma ferramenta genérica disponível em sistemas Unix-like para manipular a saída de texto.

```sh
aws logs filter-log-events --log-group-name /aws/lambda/legalone-client --filter-pattern "\"Service Unavailable\"" --start-time $(date -d '5 minutes ago' +%s)000 | jq -r '.events[].message' | grep -oE 'RequestId: [a-f0-9-]+' | sort -u | while read -r line; do request_id=$(echo "$line" | cut -d' ' -f2); echo "=== Request ID: $request_id ==="; aws logs filter-log-events --log-group-name /aws/lambda/legalone-client --filter-pattern "\"$request_id\"" --start-time $(date -d '5 minutes ago' +%s)000 | jq -r '.events[].message'; done
```
- Este comando é uma combinação de várias operações para filtrar e exibir logs relacionados a mensagens de "Service Unavailable" (Serviço Indisponível) em uma função Lambda específica.
- O comando `aws logs filter-log-events` é usado para filtrar os eventos de logs do grupo de logs `/aws/lambda/legalone-client` com o padrão de filtro `"Service Unavailable"`. A opção `--start-time $(date -d '5 minutes ago' +%s)000` define o início do período de tempo para os logs, começando 5 minutos atrás.
- A saída do comando é processada usando `jq` para extrair as mensagens dos eventos de log, e `grep` é usado para encontrar os Request IDs únicos associados a essas mensagens de "Service Unavailable". O `sort -u` é usado para garantir que apenas Request IDs únicos sejam processados.
- O comando `while read -r line; do ... done` é usado para iterar sobre cada Request ID encontrado. Para cada Request ID, ele executa outro comando `aws logs filter-log-events` para filtrar os logs relacionados a esse Request ID específico, permitindo que você veja os detalhes dos logs associados a cada ocorrência de "Service Unavailable". O resultado é exibido no terminal, mostrando os logs relevantes para cada Request ID identificado.


# Documentação do FastAPI
```sh
curl -s -w "\nHTTP %{http_code}\n" "https://vtyx2rgl2d.execute-api.us-east-1.amazonaws.com/prod/docs"
```

# Uso do Comando 'echo', 'tail' e '2>&1' no Terminal
```sh
echo "y" | sam deploy 2>&1 | tail -25
```
- O comando 'tail -25' exibe as últimas 25 linhas da saída do comando 'sam deploy'. Ele é usado para mostrar apenas as partes mais recentes da saída, o que pode ser útil para focar nas mensagens finais ou nos resultados do comando, especialmente quando a saída é longa.
- O comando 'echo' é usado para enviar a string "y" (que representa "yes") para a entrada padrão do comando 'sam deploy'. Isso é útil quando o comando 'sam deploy' solicita uma confirmação do usuário (por exemplo, "Do you want to continue? (y/n)"), permitindo que o processo continue automaticamente sem a necessidade de intervenção manual.
- O comando '2>&1' significa que a saída de erro padrão (stderr) do comando 'sam deploy' será redirecionada para a saída padrão (stdout). Isso garante que tanto as mensagens normais quanto as mensagens de erro sejam capturadas e processadas juntas, o que é especialmente útil quando você deseja analisar ou filtrar a saída do comando, como no caso do uso do 'tail', que mostra apenas as últimas linhas da resposta. Se não usado, as mensagens de erro poderiam ser perdidas.

# Busca Recursiva com Flags e Argumentos
```sh 
grep -rn "with open(" --include="*.py" --exclude-dir=.venv
```

# Comando para Rodar pytest com Flags
```sh
.venv/bin/python -m pytest tests/infrastructure/test_session_manager_concurrency.py -v --tb=short 2>&1
```
- O comando `.venv/bin/python -m pytest` é usado para executar os testes usando o framework pytest dentro de um ambiente virtual Python localizado na pasta `.venv`. O `-m pytest` indica que o módulo pytest deve ser executado.
- O argumento `tests/infrastructure/test_session_manager_concurrency.py` especifica o caminho para o arquivo de teste que deve ser executado. Neste caso, é um teste específico localizado na pasta `tests/infrastructure`.
- A flag `-v` (verbose) é usada para aumentar a verbosidade da saída do pytest, mostrando mais detalhes sobre os testes que estão sendo executados, incluindo os nomes dos testes e seus resultados.
- A flag `--tb=short` é usada para configurar o formato do traceback (tb) em caso de falha nos testes. O valor `short` indica que o traceback deve ser exibido de forma resumida, mostrando apenas as informações mais relevantes sobre a falha, o que pode ajudar a identificar rapidamente o problema sem exibir um traceback completo e detalhado. 
- O `2>&1` é usado para redirecionar a saída de erro padrão (stderr) para a saída padrão (stdout), garantindo que todas as mensagens, incluindo erros, sejam capturadas e exibidas juntas. Isso é útil para análise e depuração, especialmente quando a saída do comando é longa ou quando se deseja filtrar ou processar a saída de forma específica. O `2>` indica que estamos redirecionando o fluxo de erro (stderr), e `&1` indica que ele deve ser redirecionado para onde a saída padrão (stdout) está atualmente direcionada.

# Comando com grep e flags
```sh
grep -n "CreateTaskParserResult\|interpret_create_task_response\|task_id" /Users/thomas/Documents/projetos/legal-one-client/tests/tasks/*.py 2>/dev/null | head -60
```
- O comando `grep` é usado para procurar por padrões específicos em arquivos. Neste caso, ele está procurando pelas palavras "CreateTaskParserResult", "interpret_create_task_response" ou "task_id" dentro dos arquivos Python localizados na pasta `/Users/thomas/Documents/projetos/legal-one-client/tests/tasks/`.

- A opção `-n` faz com que o `grep` exiba o número da linha onde cada ocorrência do padrão é encontrada, o que facilita a localização das palavras dentro dos arquivos.

- O `2>/dev/null` é usado para redirecionar a saída de erro padrão (stderr) para o dispositivo nulo, o que significa que quaisquer mensagens de erro geradas pelo comando `grep` (como arquivos não encontrados ou problemas de permissão) serão descartadas e não aparecerão na saída do terminal. O `2>` indica que estamos redirecionando o fluxo de erro (stderr), e `/dev/null` é um dispositivo especial que descarta tudo o que é enviado para ele.

- O operador `|`  em `| head -60` é usado para encadear o comando `grep` com o comando `head`, permitindo que a saída do `grep` seja processada pelo `head`. O `head` é usado para limitar a quantidade de linhas exibidas, mostrando apenas as primeiras 60 linhas da saída do `grep`.

In [ ]:

# O método __all__ em Python é uma lista especial que define quais nomes serão exportados quando alguém usar from module import *.

# Finalidade Principal
# O __all__ serve como um "filtro de exportação" que controla a interface pública do seu módulo. Ele especifica exatamente quais objetos (funções, classes, variáveis) estarão disponíveis quando o módulo for importado com *.

# Como Funciona
# # Exemplo de uso correto do __all__
# from .models import User, Product
# from .utils import helper_function
# from .constants import API_URL

# # Define o que será exportado publicamente
# __all__ = [
#     'User',
#     'Product', 
#     'helper_function',
#     'API_URL'
# ]

# Exemplo Prático
# Sem __all__:
# # modulo.py
# def funcao_publica():
#     pass

# def _funcao_privada():
#     pass

# CONSTANTE = "valor"
# _variavel_interna = "secreto"

# Com __all__:
# # modulo.py
# def funcao_publica():
#     pass

# def _funcao_privada():
#     pass

# CONSTANTE = "valor"
# _variavel_interna = "secreto"

# __all__ = ['funcao_publica', 'CONSTANTE']

# Benefícios
# Controle de Interface: Define claramente o que é público vs privado
# Evita Poluição: Previne importação de objetos internos
# Documentação: Serve como documentação da API pública
# Manutenibilidade: Facilita refatoração de código interno
# ⚠️ Gotcha Importante
# O __all__ não impede importações diretas como from modulo import _funcao_privada - ele só afeta import *.




Boa pergunta. Vou explicar cada conceito do zero.

---

## O que é o `logging` do Python

É o sistema padrão de registrar mensagens da sua aplicação. Em vez de `print()`, você usa:

```python
logger.info("Contato criado com sucesso")
logger.warning("Sessão expirou, tentando renovar")
logger.error("Falha ao conectar no LegalOne")
```

Cada mensagem tem um **nível de severidade**:

| Nível | Quando usar |
|-------|-------------|
| `DEBUG` | Detalhes internos, só útil durante desenvolvimento |
| `INFO` | Eventos normais do fluxo ("início da request", "login concluído") |
| `WARNING` | Algo inesperado mas não crítico |
| `ERROR` | Algo deu errado mas a aplicação continua |
| `CRITICAL` | Falha grave, aplicação pode parar |

---

## O que é um "handler"

O handler responde à pergunta: **"para onde vai a mensagem?"**

- `StreamHandler` → escreve no terminal (stdout/stderr)
- `FileHandler` → escreve em arquivo
- `SysLogHandler` → envia para syslog

Sem handler, as mensagens simplesmente **desaparecem** — o logger não sabe para onde mandar.

---

## O que é o "root logger"

O Python tem uma hierarquia de loggers. O `root logger` é o pai de todos. Quando você faz:

```python
logging.getLogger("session_manager")
```

Esse logger "herda" o handler do root, a menos que você configure o dele próprio.

---

## O que é o `basicConfig`

É um atalho para configurar o root logger de uma vez só:

```python
logging.basicConfig(
    level=logging.INFO,               # só processa INFO pra cima (ignora DEBUG)
    format="%(asctime)s %(levelname)s %(name)s — %(message)s",
    #        data/hora    INFO/ERROR   session_manager   mensagem
)
```

Sem isso, o root logger não tem handler → mensagens somem.

---

## Por que "no-op se já tiver handlers"

`basicConfig` tem uma proteção embutida:

```python
# comportamento interno do basicConfig:
if len(root_logger.handlers) == 0:
    # configura normalmente
else:
    # não faz nada (no-op)
```

Isso importa porque:

- **No Lambda**: a AWS já configura o root logger com um `StreamHandler` apontando para stdout antes de chamar seu código. Se você chamasse `basicConfig` sem essa proteção, adicionaria um segundo handler e cada mensagem apareceria **duplicada** no CloudWatch.
- **Nos testes locais**: o pytest também configura o logging antes de rodar os testes. Sem essa proteção, `basicConfig` sobrescreveria a configuração do pytest.

---

## O que é o CloudWatch Logs

É o serviço de logs da AWS. No Lambda, **tudo que é escrito no stdout** (o terminal da função) é automaticamente capturado e enviado para o CloudWatch Logs. Por isso um `StreamHandler` apontando para stdout é suficiente — você não precisa de nenhuma integração especial com a AWS.

---

## Resumo visual do fluxo

```
seu código
  └─ logger.info("Login concluído")
       └─ root logger
            └─ StreamHandler
                 └─ stdout (terminal)
                      └─ [no Lambda] CloudWatch Logs captura automaticamente
                      └─ [local]     aparece no terminal
```

---

## Por que silenciamos boto3, botocore e urllib3

```python
logging.getLogger("boto3").setLevel(logging.WARNING)
logging.getLogger("botocore").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)
```

Essas bibliotecas são **muito verbosas** em `INFO` — cada request HTTP que o `boto3` faz (ex.: ler cookies do DynamoDB) gera dezenas de linhas de log. Rebaixar para `WARNING` faz com que só erros reais apareçam, mantendo o CloudWatch legível.

aws logs describe-log-streams \
  --log-group-name "/aws/lambda/legalone-client" \
  --order-by LastEventTime \
  --descending \
  --max-items 5 \
  --query "logStreams[*].{stream:logStreamName, last:lastEventTimestamp}" \
  --output table 2>&1

---------------------------------------------------------------------------
|                           DescribeLogStreams                            |
+---------------+---------------------------------------------------------+
|     last      |                         stream                          |
+---------------+---------------------------------------------------------+
|  1776197694812|  2026/04/14/[$LATEST]3d54fb0129564b7bb6b938df99c17ca5   |
|  1776197693427|  2026/04/14/[$LATEST]d37759dc632b4d86bf659729b7787787   |
|  1776196910058|  2026/04/14/[$LATEST]ec57fc0301554bc18407a6084066033a   |
|  1776196891107|  2026/04/14/[$LATEST]c7ba7cd7b37b4e659082479ac7c5ed4d   |
|  1776196409689|  2026/04/14/[$LATEST]b9644c4e6e56405cb96202bf3c7b1634   |
+---------------+---------------------------------------------------------+

# Comandos Git
## git log --oneline -5
- Faz uma busca no histórico de commits e exibe os 5 mais recentes em uma linha cada, mostrando o hash abreviado do commit e a mensagem associada. É útil para obter uma visão rápida dos últimos commits feitos no repositório.

## git remote -v
- Exibe os repositórios remotos configurados para o projeto, mostrando o nome do repositório e a URL associada. Isso é útil para verificar para onde os commits serão enviados (push) ou de onde serão puxados (pull).

## git branch --show-current
- Exibe o nome da branch atual em que você está trabalhando. Isso é útil para confirmarr em qual branch você está antes de fazer commits, merges ou outras operações que dependem do contexto da branch.

## git reset --soft HEAD~1
- Move o ponteiro do HEAD para o commit anterior (HEAD~1) sem alterar os arquivos no diretório de trabalho. Isso significa que o commit mais recente será desfeito, mas as alterações feitas nesse commit permanecerão como mudanças não comitadas (staged). É útil para corrigir um commit recente sem perder as alterações feitas.
- A diferença entre usar esse comando e git commit com a flag --amend é que o git reset --soft HEAD~1 desfaz o commit mais recente, mas mantém as alterações como staged, permitindo que você faça um novo commit com uma mensagem diferente ou adicione mais alterações antes de commitar novamente. Já o git commit --amend é usado para modificar o commit mais recente, permitindo que você altere a mensagem do commit ou adicione mais alterações ao mesmo commit sem criar um novo.
- quando usar cada um: 
  - Use `git reset --soft HEAD~1` quando você deseja desfazer o commit mais recente, mas manter as alterações feitas nesse commit como staged, permitindo que você faça um novo commit com uma mensagem diferente ou adicione mais alterações antes de commitar novamente.
  - Use `git commit --amend` quando você deseja modificar o commit mais recente, seja para alterar a mensagem do commit ou para adicionar mais alterações ao mesmo commit sem criar um novo. É útil para corrigir pequenos erros ou para melhorar a descrição do commit sem alterar o histórico de commits.

## git reset --hard meu-remoto/minha-branch
- O comando `git reset --soft HEAD~1` é usado para desfazer o commit mais recente, mas manter as alterações feitas nesse commit como staged. Isso significa que as mudanças ainda estarão prontas para serem comitadas novamente, permitindo que você faça um novo commit com uma mensagem diferente ou adicione mais alterações antes de commitar novamente. Use este comando quando você deseja corrigir um commit recente sem perder as alterações feitas.

## git push meu-remoto minha-branch --force
- O comando `git reset --hard meu-remoto/minha-branch` é usado para redefinir a branch local para o estado da branch remota especificada. Ele move o ponteiro do HEAD para o commit mais recente da branch remota e descarta todas as alterações locais, incluindo commits, arquivos modificados e arquivos não rastreados. Use este comando com cuidado, pois ele pode resultar na perda de trabalho local que não foi comitado.

## Quando usar loggers filhos

### Como criar

```python
# padrão universal — usar o nome do módulo
logger = logging.getLogger(__name__)
```

O `__name__` é resolvido pelo Python automaticamente para o caminho do módulo:
- em service.py → `"services.contact.service"`
- em base_crawler.py → `"infrastructure.crawler.base_crawler"`

Isso **já é o que todos os seus arquivos fazem**. Você já está usando loggers filhos corretamente.

---

### Por que é melhor que `logging.getLogger("nome_fixo")` ou `print()`

```python
# ❌ nome fixo — dificulta rastrear de onde veio a mensagem
logger = logging.getLogger("meu_logger")

# ❌ print — sem nível, sem timestamp, sem filtro, sem handler configurável
print("contato criado")

# ✅ __name__ — hierarquia automática, rastreável, configurável
logger = logging.getLogger(__name__)
```

---

### O principal benefício: controle granular por namespace

A hierarquia permite **silenciar ou amplificar** partes específicas do sistema sem tocar no código:

```python
# silenciar só o crawler em produção (muito verboso)
logging.getLogger("infrastructure.crawler").setLevel(logging.WARNING)

# amplificar só o service de contato para depurar um bug específico
logging.getLogger("services.contact").setLevel(logging.DEBUG)

# deixar o resto em INFO normalmente
logging.getLogger().setLevel(logging.INFO)
```

Útil no seu caso porque:
- `infrastructure.crawler.base_crawler` faz muitas requests → pode ser verboso em DEBUG
- `infrastructure.crawler.session_manager` tem fluxo crítico → sempre quer INFO
- `services.contact.service` tem dados sensíveis (CPF) → pode querer nível diferente por ambiente

---

### Cenários estratégicos

**1. Ambiente diferente por namespace (dev vs. produção)**

```python
# config.py — carregado no startup
if ENV == "production":
    logging.getLogger("infrastructure.crawler").setLevel(logging.WARNING)
    logging.getLogger("services").setLevel(logging.INFO)
else:
    logging.getLogger().setLevel(logging.DEBUG)  # tudo visível em dev
```

**2. Handler separado para erros críticos**

```python
# erros do crawler vão para um SNS/Slack além do CloudWatch
crawler_logger = logging.getLogger("infrastructure.crawler")
crawler_logger.addHandler(SlackAlertHandler())  # só para esse namespace
```

**3. Rastrear origin no CloudWatch Logs Insights**

Como cada log registra o nome do logger (`%(name)s`), você consegue filtrar:

```
# CloudWatch Logs Insights
fields @message
| filter @logStream like /legalone-client/
| filter logger = "infrastructure.crawler.session_manager"
| filter level = "ERROR"
```

**4. Testes — silenciar logs desnecessários**

```python
# conftest.py
@pytest.fixture(autouse=True)
def silenciar_crawler_logs():
    logging.getLogger("infrastructure.crawler").setLevel(logging.CRITICAL)
```

---

### Quando **não** criar um logger filho separado

| Situação | Por quê não |
|----------|-------------|
| Funções utilitárias puras (`_strip_tags`, `_decode_html_entities`) | Não fazem I/O, não falham em produção — `logger` seria nunca usado |
| Payload builders | Montagem pura de dados — erros são bugs de código, não eventos de runtime |
| Schemas Pydantic | Validação já levanta `ValueError` com mensagem clara |

---

### Resumo prático para o seu projeto

```
root logger  (setLevel INFO no main.py)
│
├── infrastructure.crawler.base_crawler      → WARNING em prod (verboso)
├── infrastructure.crawler.session_manager   → INFO sempre (fluxo crítico)
├── infrastructure.crawler.contacts          → INFO
├── infrastructure.crawler.tasks             → INFO
│
├── services.contact.service                 → INFO
├── services.task.task_service               → INFO
│
├── parsers.contact_parser                   → ERROR (só falhas de parse)
├── parsers.task_parser                      → ERROR
│
└── app.error_handler                        → WARNING+ (já captura tudo)
```

O que você já tem está correto — `logging.getLogger(__name__)` em cada módulo é exatamente a boa prática. O único ajuste estratégico seria configurar níveis por namespace no main.py conforme o ambiente.